### PHASE 4: The Scikit-Learn Business Implementation
**Implementation pattern strictly followed:** X/y $\rightarrow$ split $\rightarrow$ preprocessing $\rightarrow$ baseline $\rightarrow$ fit $\rightarrow$ probability predictions $\rightarrow$ threshold $\rightarrow$ class predictions $\rightarrow$ metrics $\rightarrow$ error analysis[cite: 2].

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

print("\n--- PHASE 4: CUSTOMER CHURN PREDICTION PIPELINE ---")
# 1. Dataset (Simulating Customer churn prediction[cite: 2])
np.random.seed(42)
n_samples = 1000
hours_played = np.random.uniform(5, 100, n_samples)
days_inactive = np.random.uniform(0, 30, n_samples)

# Under the hood: Log-odds transform probability into an unbounded linear quantity[cite: 2].
# More days inactive increases log-odds of churn; more hours played decreases it.
simulated_log_odds = -2.0 + (days_inactive * 0.2) - (hours_played * 0.05)
simulated_probs = sigmoid_by_hand(simulated_log_odds)
y = (np.random.rand(n_samples) < simulated_probs).astype(int)
X = np.column_stack((hours_played, days_inactive))

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Fit Baseline Model
# Logistic regression is a strong baseline because it is fast, relatively interpretable, and produces probabilities[cite: 2].
# (Note: Multiclass classification can be handled with appropriate multiclass strategies[cite: 2], but we are doing binary here).
# Regularization controls model complexity[cite: 2]. (scikit-learn applies Ridge/L2 by default).
baseline = LogisticRegression(C=1.0)
baseline.fit(X_train, y_train)

# 4. Inspect coefficients and their direction[cite: 2]
print(f"Feature 1 (Hours) Coefficient: {baseline.coef_[0][0]:.4f} (Negative: Drops churn risk)")
print(f"Feature 2 (Inactive) Coefficient: {baseline.coef_[0][1]:.4f} (Positive: Raises churn risk)")

# Cell 7: Python Code (PHASE 4: Thresholds & Confusion Matrices)

In [ ]:
print("\n--- PHASE 4 DELIVERABLE: PROBABILITY VS THRESHOLD EXPERIMENT ---")

# Compare probability outputs with hard class outputs[cite: 2].
y_prob = baseline.predict_proba(X_test)[:, 1] # Extract just the probability of Churn=1

# We will change the classification threshold without retraining[cite: 2].
test_thresholds = [0.3, 0.5, 0.8]

# We are outputting a Confusion matrix at multiple thresholds[cite: 2].
for t in test_thresholds:
    # Probability is continuous; class prediction is obtained by applying a decision threshold[cite: 2].
    y_pred_adjusted = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_adjusted)
    print(f"\nThreshold set to {t}:")
    print(cm)

# Cell 8: Python Code (PHASE 4: The Overfitting Trap of Trees)

In [ ]:
print("\n--- PHASE 4 DELIVERABLE: TREE COMPLEXITY & BIAS/VARIANCE ---")
# You should understand the relationship between: Tree depth -> model complexity -> bias/variance[cite: 2].

depths_to_test = [1, 5, 25]
for d in depths_to_test:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    
    print(f"Tree Depth {d} | Train Accuracy: {train_acc*100:.1f}% | Test Accuracy: {test_acc*100:.1f}%")

print("\nNotice that at Depth 25, Training Accuracy hits 100%, but Test Accuracy drops!")
print("Increasing tree depth generally decreases bias, but can cause variance to increase, which can lead to overfitting[cite: 2].")